# Q1 — o tamanho da diferença

Objetivo: Investigar a diferença nas notas da prova de Ciências da Natureza (CN) do ENEM entre as redes de ensino, faixas de renda per capita e níveis de escolaridade dos pais. Essa diferença muda ao longo do tempo?

- **Por renda e escolaridade dos pais:** só entre os anos de 2020-2023 (`fct_individual`), porque em 2024-2025 a nota não se liga mais ao questionário socioeconômico, como explicado pela D002.

- **Por rede**: 2020-2025 (`agg_rede_ano`), com atenção à quebra de série entre 2023 e 2024, como explicada pela D002.

## Configurações iniciais

In [ ]:
import sys
from src.db import conectar
con = conectar(read_only=True)
from src import graficos as g
g.aplicar_tema()


## Análise entre as diferentes redes de ensino

Começando com a análise entre as redes de ensino, vamos utilizar a *tabela* `agg_rede_ano`, selecionando as colunas que mostram o `ano`, o período (`periodo`), a `rede`, a quantidade de participantes com nota de Ciências da Natureza (`qtd_com_nota_cn`), a média da nota de Ciências da Natureza (`media_nota_cn`), a mediana (`p50_nota_cn`) e o desvio padrão da nota de Ciências da Natureza (`desvio_padrao_nota_cn`) ordenadas pela `rede` e pelo `ano`.

In [ ]:
rede = con.sql("""
    SELECT ano, periodo, rede,
        qtd_com_nota_cn, media_nota_cn, p50_nota_cn,
        desvio_padrao_nota_cn
    FROM agg_rede_ano
    ORDER BY rede, ano
""").df()
rede

### A quebra de série entre 2023-2024

Como explicado na D002, entre 2023-2024, o INEP alterou a forma como as informações sobre a escola do participante são obtidas. Até 2023, era de responsabilidade do participante preencher as informações e, a partir de 2024, essas informações passaram a ser obtidas através do Censo Escolar utilizando o CPF do participante.

Como efeito imediato dessa mudança, há uma quebra de série visível nos dados, principalmente para as escolas **municipais** e **estaduais** (`rede=estadual_municipal`). Essa mudança pode ser vista com clareza na coluna `qtd_com_nota_cn`:

| Rede | 2023 | 2024 | Variação |
|------------|----------|---------|---------|
|Estadual/Municipal| 28.887 | 73.206 | +153% (2,5 vezes) |
| Federal | 5.666 | 7.371 | +30% |
|Privada | 18.404 | 21.432 | +16%|

A rede estadual/municipal mais que dobrou enquanto as outras tiveram crescimento entre 16% e 30%.

### A nota média por rede de ensino

Vamos montar um gráfico para analisar a nota média de Ciências da Natureza ao longo do tempo, levando em consideração a quebra de série descrita anteriormente:

In [ ]:
fig, ax = g.serie_temporal(
    rede, x="ano", y="media_nota_cn", grupo="rede",
    rotulos={"federal": "Federal",
            "estadual_municipal": "Estadual/municipal",
            "privada": "Privada"},
    marcar_quebra=True,
    rotulo_quebra="Mudança na fonte da escola",
    conectar_quebra=False,
    legenda=False,
    titulo="Média em Ciências da Natureza por rede de ensino — MG, 2020–2025",
    subtitulo="Concluintes de escolas mineiras com nota na prova",
    rotulo_y="Nota média (pontos)",
    nota="Fonte: microdados do ENEM (INEP). Até 2023, a escola era informada "
        "pelo participante; a partir de 2024, vem do Censo Escolar.",
    figsize=(8, 5),
)
g.salvar(fig, "q1_media_cn_por_rede")

In [ ]:
var_anos = (rede[rede["periodo"] == "2020-2023"]
            .groupby("rede")["media_nota_cn"]
            .agg(desvio_padrao="std",
                minimo="min",
                maximo="max")
            .assign(amplitude=lambda d: d["maximo"] - d["minimo"])
            .round(1))
print(var_anos.to_string())

Entre 2020 e 2023, a variação das notas pode ser melhor observada na tabela:

| Rede | Faixa em 2020-2023 | Amplitude | Desvio Padrão da média |
|-------|------------|-----------|-----------|
|Federal| de 551,0 (2023) a 569,4 (2020)| 18,4 pontos| 8,2|
|Privada|de 553,6 (2022) a 570,3 (2020)| 16,7 pontos| 8,0|
|Estadual/municipal| de 475,0 (2021) a 488,3 (2023)| 13,2 pontos| 6,1|

A rede com menor variação, tanto da amplitude quanto do desvio padrão da média no período é a Estadual/municipal e a rede com maior variação é a Federal. 

A quebra de série entre 2023 e 2024 pode explicar a queda da nota média entre esses mesmos anos para a rede estadual/municipal. Como a quantidade de participantes mais que dobrou entre os anos, podemos entender a queda como um efeito de composição da amostra e não como uma piora real do resultado dos estudantes. Já o efeito é bem menos sentido para as redes federal e privada, onde o aumento do número de participantes teve um crescimento relativamente menor (entre 16% e 30%).

Vamos comparar agora as notas entre as três redes de ensino ano por ano:

In [ ]:
gaps = (rede.pivot(index="ano", columns="rede", values="media_nota_cn")
        .assign(privada_menos_estadual=lambda d: d["privada"] - d["estadual_municipal"],
                federal_menos_estadual=lambda d: d["federal"] - d["estadual_municipal"],
                privada_menos_federal=lambda d: d["privada"] - d["federal"])
        .round(1))
print(gaps.to_string())

||média estadual/municipal| média federal| média privada| privada menos estadual| federal menos estadual| privada menos federal|
|---------|--------|---------|----------|---------|---------|-------|
|2020|486,3|569,4|570,3|84,0|83,1|0,9|
|2021|475,0|556,8|559,2|84,1|81,8|2,3|
|2022|486,9|553,3|553,6|66,7|66,3|0,3|
|2023|488,3|551,0|569,0|80,8|62,8|18,0|
|2024|475,2|552,7|568,4|93,2|77,5|15,7|
|2025|479,8|562,9|571,5|91,7|83,1|8,6|


O gap entre a rede federal e a rede privada é quase zero entre 2020 e 2022, as duas linhas praticamente se sobrepõe no gráfico, mas a separação entre elas fica mais evidente em 2023, onde a média da rede federal continua diminuindo e a da rede privada aumenta. Nesse ano, o gap entre as duas passa a ser de 18,0 pontos. Como isso acontece antes da quebra de série, o motivo por trás é outro, mas por enquanto não consegui formular nenhuma hipótese a respeito.

As médias da rede estadual/municipal seguem uma crescente no período de 2021-2023, o que pode indicar um melhor desempenho da rede, mas mais análises precisam ser feitas para entender melhor essa hipótese.

Em 2020, a média das três redes assume um valor maior se comparado com a média do ano seguinte. O ano de 2020 foi um ano de abstenção recorde (55%) devido à pandemia de COVID-19. As médias de 2020 podem ter sido infladas se, de alguma forma, um viés de seleção foi aplicado aos participantes que compareceram para a realização da prova.



A média e a mediana (`p_50_nota_cn`) das notas de Ciências da Natureza apresentam pouca variação para a rede estadual/municipal, mas na maioria dos casos a média é maior que a mediana. Já para as redes privada e federal, a mediana é maior que a média em todos os anos analisados.

Para o caso da rede estadual/municipal, a assimetria evidencia uma cauda de notas mais altas para a maioria dos anos analisados puxando a média para cima e no caso das redes privada e federal, a assimetria mostra uma cauda de notas mais baixas para todos os anos analisados puxando a média para baixo.



In [ ]:
rotulos_rede = {"federal": "Federal",
                "estadual_municipal": "Estadual/municipal",
                "privada": "Privada"}

notas = con.sql("""
    SELECT ano, rede, nota_cn
    FROM fct_agregada
    WHERE tem_nota_cn AND ano IN (2023, 2025)
""").df()

for ano in (2023, 2025):
    fig, ax = g.distribuicoes(
        notas[notas["ano"] == ano],
        coluna_valor="nota_cn",
        coluna_grupo="rede",
        rotulos=rotulos_rede,
        mostrar_n=True,
        intervalo=(290, 850),
        preencher=False,
        titulo=f"Distribuição das notas em Ciências da Natureza por rede — MG, {ano}",
        subtitulo="Concluintes de escolas mineiras com nota na prova",
        rotulo_x="Nota em Ciências da Natureza (pontos)",
        rotulo_y="Densidade",
        nota="Fonte: microdados do ENEM (INEP). Curvas normalizadas: cada rede tem área 1.",
    )



In [ ]:
con.sql("""
    SELECT ano, rede,
           COUNT(*) FILTER (WHERE nota_cn = 0) AS nota_zero,
           COUNT(*) FILTER (WHERE nota_cn > 0 AND nota_cn < 300) AS abaixo_300,
           COUNT(*) AS com_nota,
           MIN(nota_cn) FILTER (WHERE nota_cn > 0) AS menor_nota_positiva
    FROM fct_agregada
    WHERE tem_nota_cn
    GROUP BY ALL ORDER BY ano, rede
""").show(max_rows=50)

In [ ]:
con.sql("""
    SELECT ano, vetor_respostas_cn
    FROM fct_agregada
    WHERE tem_nota_cn AND nota_cn = 0
    LIMIT 10
""").show()

In [ ]:
con.sql("""
    SELECT ano, COUNT(*) FILTER (WHERE nota_cn = 0) AS nota_zero
    FROM fct_individual WHERE tem_nota_cn
    GROUP BY ano ORDER BY ano
""").show()

In [ ]:
con.sql("""
    SELECT codigo_prova_cn,
           COUNT(*) FILTER (WHERE nota_cn = 0) AS nota_zero,
           COUNT(*) AS com_nota
    FROM fct_individual
    WHERE ano = 2023 AND tem_nota_cn
    GROUP BY codigo_prova_cn
    HAVING COUNT(*) FILTER (WHERE nota_cn = 0) > 0
    ORDER BY nota_zero DESC
""").show()